# Models

Three analyses, one pipeline each:

1. **Union density** (blue charts)
2. **Electoral democracy** (orange charts)
3. **Electoral democracy without V-Dem predictors** (green charts)

Each section fits an OLS on the training countries for its coefficients, then scores OLS and a random forest with **nested**, grouped cross-validation: the forest is re-tuned inside every outer fold, using only that fold's training countries, so no country it is scored on ever influenced its settings. Permutation importances come from the same folds.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
# OLS is imported directly: statsmodels.api also loads statsmodels.tsa.statespace,
# whose compiled _conventional DLL is blocked by Windows Application Control
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
# NOTE: tuning.ipynb has an identical copy of this block
df = pd.read_csv("../data/merged.csv")

ID_COLS = ["iso3", "country", "Unnamed: 0"]
DROP = ["emp_services_pct",     # emp shares sum to exactly 100
        "services_va_pct_gdp",  # value-added shares nearly sum to 100
        "emp_pop_ratio",        # ≈ lfp_rate × (1 − unemployment)
        "self_employed_pct",    # r = 0.996 with vulnerable_emp_pct
        "exports_pct_gdp",      # r = 0.987 with trade_pct_gdp
        "liberal_democracy"]    # r = 0.981 with electoral_democracy
# V-Dem builds electoral_democracy (v2x_polyarchy) from freedom_of_association
# (v2x_frassoc_thick), which is itself built from cso_entry_exit and
# cso_repression; civil_society_index shares those two inputs (codebook v16).
# Predicting democracy from them would be predicting the index from its parts
DEM_INGREDIENTS = ["freedom_of_association", "cso_entry_exit",
                   "cso_repression", "civil_society_index"]
# every V-Dem column in merged.csv (VDEM_NAMES in cleaning.ipynb), for the
# democracy model with no V-Dem predictors at all
VDEM_COLS = ["freedom_of_association", "civil_society_index", "electoral_democracy",
             "liberal_democracy", "cso_repression", "cso_entry_exit",
             "egalitarian_component", "equal_resource_distribution",
             "power_by_socioeconomic_position", "state_mass_org_engagement",
             "regime_support_working_class", "govt_ideology_socialist"]
# predictors from the union-density literature -- Ghent system, state-run vs
# independent unions, labour's political power -- used by the union density
# model only
UNION_ONLY = ["ghent", "state_mass_org_engagement",
              "regime_support_working_class", "govt_ideology_socialist"]

cols_union = [c for c in df if c not in ID_COLS + ["union_density"] + DROP]
cols_dem = [c for c in df if c not in ID_COLS + ["electoral_democracy"] + DROP + DEM_INGREDIENTS + UNION_ONLY]
cols_dem_novdem = [c for c in df if c not in ID_COLS + VDEM_COLS + DROP + UNION_ONLY]

# rows must be complete in every column except the union-only additions, so
# adding them can't change the democracy analyses or the train/test split. the
# union model drops its own incomplete rows, if any
df = df.dropna(subset=[c for c in df.columns if c not in UNION_ONLY]).reset_index(drop=True)

random_state = 42

# one train/test split: models.ipynb fits its OLS summaries on the training
# rows, and tuning.ipynb tunes on the training countries only. whole countries
# are held out, so no country appears in both halves
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
train_idx, test_idx = next(splitter.split(df, groups=df["iso3"]))
train, test = df.loc[train_idx], df.loc[test_idx]

print(f"train: {len(train):4d} rows, {train['iso3'].nunique():3d} countries")
print(f"test:  {len(test):4d} rows, {test['iso3'].nunique():3d} countries")
print("countries in both:", len(set(train["iso3"]) & set(test["iso3"])))
print(f"\nfeatures -- union density: {len(cols_union)} | democracy: {len(cols_dem)} | "
      f"democracy, no V-Dem: {len(cols_dem_novdem)}")

train:  787 rows,  80 countries
test:   208 rows,  20 countries
countries in both: 0

features -- union density: 39 | democracy: 31 | democracy, no V-Dem: 28


In [3]:
# shared by every section, so each analysis gets exactly the same models,
# scoring, and charts
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#008300"   # sections 1, 2, 3
MUTED, SURFACE = "#52514e", "#fcfcfb"
CV_REPEATS, CV_FOLDS = 3, 5

# the random forest's settings are searched over this grid (the same one as
# tuning.ipynb) inside every outer CV fold
GRID = {
    "max_features": [0.2, 0.33, 0.5, 1.0],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_samples": [None, 0.7],
}
SEARCH_TREES, FINAL_TREES = 300, 1000


def scale(cols):
    """Scaled training features for the OLS summaries, on the training rows
    that are complete for these columns."""
    rows = train.dropna(subset=cols)
    scaler = StandardScaler().fit(rows[cols])
    return pd.DataFrame(scaler.transform(rows[cols]), columns=cols, index=rows.index)


def tuned_forest(X, y, groups):
    """Grid-search the forest on these rows only, with inner folds grouped by
    country, then refit the winning settings with more trees."""
    search = GridSearchCV(
        RandomForestRegressor(n_estimators=SEARCH_TREES, random_state=random_state, n_jobs=1),
        GRID,
        cv=GroupKFold(CV_FOLDS),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    ).fit(X, y, groups=groups)
    forest = RandomForestRegressor(n_estimators=FINAL_TREES, random_state=random_state,
                                   n_jobs=-1, **search.best_params_)
    return forest.fit(X, y), search.best_params_


def nested_cv(X, y, groups):
    """Scores and permutation importances from nested, grouped cross-validation.

    Outer loop: 5 folds of whole countries, repeated with the countries
    reshuffled, so every country is tested once per repeat. A single
    train/test split would rest on 20 test countries, which is far too few --
    the same model scored anywhere from -2.06 to 0.50 depending on the draw.

    Inner loop: the forest is tuned on the outer fold's training countries
    only, so the countries it is scored on never influenced its settings
    (Neunhoeffer & Sternberg 2019). OLS and the baseline have nothing to tune.
    """
    names = ["baseline", "OLS", "random forest"]
    rows, importances, chosen = [], [], []
    for seed in range(CV_REPEATS):
        oof = {name: np.zeros(len(y)) for name in names}
        # same folds for every model, so the comparison is paired
        cv = GroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
        for tr, te in cv.split(X, y, groups):
            X_tr, y_tr, X_te, y_te = X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te]
            forest, best = tuned_forest(X_tr, y_tr, groups.iloc[tr])
            chosen.append(str(best))

            fitted = {
                "baseline": DummyRegressor(strategy="mean").fit(X_tr, y_tr),
                "OLS": make_pipeline(StandardScaler(), LinearRegression()).fit(X_tr, y_tr),
                "random forest": forest,
            }
            for name, model in fitted.items():
                oof[name][te] = model.predict(X_te)

            # importance measured on countries the tuned forest never saw
            r = permutation_importance(forest, X_te, y_te, n_repeats=10,
                                       scoring="neg_root_mean_squared_error",
                                       random_state=0, n_jobs=-1)
            importances.append(pd.Series(r.importances_mean, index=X.columns))

        # pool the out-of-fold predictions and score once per repeat --
        # averaging each fold's R2 instead is dragged down by folds with
        # little spread to explain
        for name in names:
            rows.append({
                "model": name,
                "seed": seed,
                "RMSE": root_mean_squared_error(y, oof[name]),
                "MAE": mean_absolute_error(y, oof[name]),
                "R2": r2_score(y, oof[name]),
            })

    scores = pd.DataFrame(rows)
    table = (scores.groupby("model")[["RMSE", "MAE", "R2"]]
             .agg(["mean", "std"]).round(3)
             .reindex(names))
    print(f"nested {CV_FOLDS}-fold grouped CV, {CV_REPEATS} repeats, "
          f"{len(X)} rows / {groups.nunique()} countries\n")
    print(table.to_string())

    wide = scores.pivot(index="seed", columns="model", values="R2")
    print(f"\nforest beat OLS in {(wide['random forest'] > wide['OLS']).sum()} of {CV_REPEATS} repeats")

    picks = pd.Series(chosen).value_counts()
    print(f"\nforest settings chosen in the {len(chosen)} outer folds "
          f"({len(picks)} different combinations); most common:")
    print(picks.head(3).to_string())
    return scores, pd.DataFrame(importances)


def importance_table(imp):
    return (pd.DataFrame({"mean": imp.mean(), "sd": imp.std(), "folds_positive": (imp > 0).sum()})
            .sort_values("mean", ascending=False))


def importance_plot(imp, color, title, unit, top_n=10):
    """One dot per CV fold, a diamond for the mean, and the number of folds
    where shuffling the feature made predictions worse."""
    stats = importance_table(imp)
    top = stats.head(top_n).index[::-1]   # largest ends up at the top

    fig = go.Figure()
    for feat in top:
        fig.add_scatter(x=imp[feat], y=[feat] * len(imp), mode="markers", showlegend=False,
                        marker=dict(size=9, color=color, opacity=0.35,
                                    line=dict(width=1, color=SURFACE)),
                        hovertemplate="%{y}<br>fold: %{x:.3f}<extra></extra>")

    fig.add_scatter(x=stats.loc[top, "mean"], y=list(top), mode="markers", showlegend=False,
                    marker=dict(size=14, color=color, symbol="diamond",
                                line=dict(width=1.5, color=SURFACE)),
                    hovertemplate="%{y}<br><b>mean: %{x:.3f}</b><extra></extra>")

    fig.add_vline(x=0, line_width=1.5, line_color=MUTED)
    for feat in top:
        fig.add_annotation(x=1.02, xref="paper", y=feat, showarrow=False,
                           text=f"{stats.loc[feat, 'folds_positive']}/{len(imp)}",
                           font=dict(size=11, color=MUTED))

    fig.update_layout(
        title=f"{title}: permutation importance across {len(imp)} grouped CV folds"
              f"<br><sub>{unit} when the feature is shuffled · diamond = mean · "
              "right column = folds where shuffling hurt</sub>",
        xaxis=dict(title=unit, gridcolor="#eceae5", zeroline=False),
        yaxis=dict(gridcolor="rgba(0,0,0,0)"),
        plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
        margin=dict(l=210, r=70, t=80, b=50), height=430,
    )
    return fig

## 1. Predicting union density

Besides the economic and V-Dem features, this section uses four predictors taken from the union-density literature (`UNION_ONLY`; sections 2 and 3 don't use them):

| Column | Source | Stands in for |
|---|---|---|
| `ghent` | hand-coded: Denmark, Finland, Iceland, Sweden, Belgium | Ghent system — union-linked unemployment insurance |
| `state_mass_org_engagement` | V-Dem `v2castate` | state-run unionism, as opposed to independent unions |
| `regime_support_working_class` | V-Dem `v2regsupgroups_9` | labour's political power |
| `govt_ideology_socialist` | V-Dem `v2exl_legitideolcr_1` | left government (partial proxy) |

In [4]:
X_train_u = scale(cols_union)
y_train_u = train.loc[X_train_u.index, "union_density"]

In [5]:
ols_union = OLS(y_train_u, add_constant(X_train_u)).fit(
    cov_type="cluster", cov_kwds={"groups": train.loc[X_train_u.index, "iso3"]}
)
print(ols_union.summary())

                            OLS Regression Results                            
Dep. Variable:          union_density   R-squared:                       0.798
Model:                            OLS   Adj. R-squared:                  0.787
Method:                 Least Squares   F-statistic:                     178.4
Date:                Sat, 03 Oct 2026   Prob (F-statistic):           5.23e-63
Time:                        19:14:34   Log-Likelihood:                -2721.6
No. Observations:                 787   AIC:                             5523.
Df Residuals:                     747   BIC:                             5710.
Df Model:                          39                                         
Covariance Type:              cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

In [6]:
union_df = df.dropna(subset=cols_union)
cv_union, imp_union = nested_cv(union_df[cols_union], union_df["union_density"], union_df["iso3"])

nested 5-fold grouped CV, 3 repeats, 995 rows / 100 countries

                 RMSE            MAE            R2       
                 mean    std    mean    std   mean    std
model                                                    
baseline       20.332  0.186  15.814  0.156 -0.052  0.019
OLS            13.698  0.912  11.209  0.602  0.521  0.064
random forest  12.173  0.556   9.867  0.368  0.623  0.035

forest beat OLS in 3 of 3 repeats

forest settings chosen in the 15 outer folds (7 different combinations); most common:
{'max_features': 0.5, 'max_samples': None, 'min_samples_leaf': 2}    4
{'max_features': 0.5, 'max_samples': 0.7, 'min_samples_leaf': 2}     4
{'max_features': 0.2, 'max_samples': 0.7, 'min_samples_leaf': 1}     2


In [7]:
importance_table(imp_union).round(3)

,mean,sd,folds_positive
ghent,3.014,2.471,14
gini,0.715,0.576,13
educ_spend_pct_gdp,0.706,0.600,14
tax_revenue_pct_gdp,0.608,0.425,14
egalitarian_component,0.424,0.370,14
labor_force_total,0.248,0.331,13
power_by_socioeconomic_position,0.232,0.193,14
equal_resource_distribution,0.188,0.169,14
manufacturing_va_pct_gdp,0.161,0.162,14
civil_society_index,0.098,0.180,9


In [8]:
importance_plot(imp_union, BLUE, "Union density", "RMSE points lost")

## 2. Predicting electoral democracy

Same models and charts as section 1, with V-Dem's `electoral_democracy` (0–1) as the target and `union_density` as one of the predictors. The four `DEM_INGREDIENTS` features are left out because V-Dem builds `electoral_democracy` from them.

In [9]:
X_train_d = scale(cols_dem)
y_train_d = train["electoral_democracy"]

In [10]:
ols_dem = OLS(y_train_d, add_constant(X_train_d)).fit(
    cov_type="cluster", cov_kwds={"groups": train["iso3"]}
)
print(ols_dem.summary())

                             OLS Regression Results                            
Dep. Variable:     electoral_democracy   R-squared:                       0.893
Model:                             OLS   Adj. R-squared:                  0.888
Method:                  Least Squares   F-statistic:                     118.3
Date:                 Sat, 03 Oct 2026   Prob (F-statistic):           2.47e-54
Time:                         19:19:42   Log-Likelihood:                 959.52
No. Observations:                  787   AIC:                            -1855.
Df Residuals:                      755   BIC:                            -1706.
Df Model:                           31                                         
Covariance Type:               cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------


In [11]:
cv_dem, imp_dem = nested_cv(df[cols_dem], df["electoral_democracy"], df["iso3"])

nested 5-fold grouped CV, 3 repeats, 995 rows / 100 countries

                RMSE           MAE            R2       
                mean    std   mean    std   mean    std
model                                                  
baseline       0.212  0.001  0.174  0.001 -0.032  0.015
OLS            0.118  0.026  0.081  0.008  0.667  0.148
random forest  0.131  0.004  0.087  0.003  0.603  0.026

forest beat OLS in 1 of 3 repeats

forest settings chosen in the 15 outer folds (10 different combinations); most common:
{'max_features': 0.2, 'max_samples': None, 'min_samples_leaf': 2}    4
{'max_features': 0.2, 'max_samples': 0.7, 'min_samples_leaf': 1}     3
{'max_features': 0.5, 'max_samples': None, 'min_samples_leaf': 1}    1


In [12]:
importance_table(imp_dem).round(3)

,mean,sd,folds_positive
egalitarian_component,0.046,0.030,15
power_by_socioeconomic_position,0.009,0.004,15
gdp_pc_const,0.005,0.005,12
emp_agriculture_pct,0.004,0.007,13
equal_resource_distribution,0.003,0.004,13
emp_industry_pct,0.003,0.002,15
health_spend_pct_gdp,0.002,0.004,10
agriculture_va_pct_gdp,0.002,0.004,9
gini,0.002,0.003,10
migrant_stock_pct,0.001,0.003,13


In [13]:
importance_plot(imp_dem, ORANGE, "Electoral democracy", "RMSE lost (index, 0–1)")

## 3. Predicting electoral democracy without V-Dem predictors

Section 2 again, but with every V-Dem column (`VDEM_COLS`) removed from the predictors — including the three equality indices, which carried most of section 2's importance and are themselves V-Dem democracy indices rated by the same experts. What's left is World Bank variables plus `union_density`, so this is the cleanest test of whether economic structure and unions predict democracy.

In [14]:
X_train_nv = scale(cols_dem_novdem)
y_train_nv = train["electoral_democracy"]

In [15]:
ols_novdem = OLS(y_train_nv, add_constant(X_train_nv)).fit(
    cov_type="cluster", cov_kwds={"groups": train["iso3"]}
)
print(ols_novdem.summary())

                             OLS Regression Results                            
Dep. Variable:     electoral_democracy   R-squared:                       0.723
Model:                             OLS   Adj. R-squared:                  0.712
Method:                  Least Squares   F-statistic:                     20.88
Date:                 Sat, 03 Oct 2026   Prob (F-statistic):           4.54e-26
Time:                         19:23:51   Log-Likelihood:                 585.83
No. Observations:                  787   AIC:                            -1114.
Df Residuals:                      758   BIC:                            -978.3
Df Model:                           28                                         
Covariance Type:               cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const     

In [16]:
cv_novdem, imp_novdem = nested_cv(df[cols_dem_novdem], df["electoral_democracy"], df["iso3"])

nested 5-fold grouped CV, 3 repeats, 995 rows / 100 countries

                RMSE           MAE            R2       
                mean    std   mean    std   mean    std
model                                                  
baseline       0.212  0.001  0.174  0.001 -0.032  0.015
OLS            0.170  0.014  0.120  0.003  0.328  0.115
random forest  0.151  0.002  0.102  0.001  0.474  0.015

forest beat OLS in 3 of 3 repeats

forest settings chosen in the 15 outer folds (9 different combinations); most common:
{'max_features': 0.2, 'max_samples': 0.7, 'min_samples_leaf': 1}     5
{'max_features': 0.2, 'max_samples': 0.7, 'min_samples_leaf': 2}     2
{'max_features': 0.2, 'max_samples': None, 'min_samples_leaf': 1}    2


In [17]:
importance_table(imp_novdem).round(3)

,mean,sd,folds_positive
gdp_pc_const,0.014,0.011,13
health_spend_pct_gdp,0.007,0.007,13
agriculture_va_pct_gdp,0.006,0.006,14
emp_agriculture_pct,0.003,0.005,13
emp_industry_pct,0.001,0.001,13
unemployment_rate,0.000,0.001,11
vulnerable_emp_pct,0.000,0.002,9
year,0.000,0.000,13
inflation_cpi,0.000,0.000,14
educ_spend_pct_gdp,0.000,0.001,8


In [18]:
importance_plot(imp_novdem, GREEN, "Electoral democracy, no V-Dem", "RMSE lost (index, 0–1)")